In [22]:
import sys
import warnings
import numpy as np
import pathlib as pl
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.stats import norm
from scipy.optimize import minimize
root_folder = pl.Path.cwd().parents[2]
sys.path.insert(0, str(root_folder / "utilities"))
import common_functions as cf

warnings.filterwarnings('ignore')

initial_data_folder = "data/initial_data/function_5"
initial_inputs_path = pl.Path.joinpath(root_folder, initial_data_folder,  "initial_inputs.npy")
initial_outputs_path = pl.Path.joinpath(root_folder, initial_data_folder, "initial_outputs.npy")

In [23]:
data_in = np.load(initial_inputs_path)
data_out = np.load(initial_outputs_path)

Week-01

In [24]:
X_init = data_in
y_init = data_out


kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(X_init, y_init)

# Surrogate to maximise (negative for minimize)
def surrogate_neg(x):
    return -gp.predict(x.reshape(1, -1))[0]

# Bounds for normalized inputs
bounds = [(0,1), (0,1), (0,1), (0,1)]

# Try multiple random starts to avoid local issues
best_x = None
best_val = float('inf')
for _ in range(10):
    x0 = np.random.rand(4)
    res = minimize(surrogate_neg, x0=x0, bounds=bounds, method='L-BFGS-B')
    if res.fun < best_val:
        best_val = res.fun
        best_x = res.x

x_next = best_x
print("Next point to evaluate:", x_next)


Next point to evaluate: [0.28524867 0.28016717 0.68455269 0.51233695]


Week-02

In [ ]:
X_init = data_in           # shape (n_samples, 4)
y_init = data_out          # shape (n_samples,)

# --- New evaluated point ---
x_new = np.array([0.232877,0.841416,0.883342,0.879464])
y_new = 1091.3271430129832

# --- Add new data to dataset ---
X_all = np.vstack([X_init, x_new])
y_all = np.hstack([y_init, y_new])

new_points = np.array([
    [0.232877,0.841416,0.883342,0.879464]
])

new_outputs = np.array([
    1091.3271430129832
])


# Combine all data
X_all = np.vstack([X_init, new_points])
y_all = np.concatenate([y_init, new_outputs])


# --- Refit Gaussian Process ---
kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(X_all, y_all)

# --- Generate candidate next points around current best ---
best_x = x_new
sigma = 0.02  # tweak size for local exploration
num_candidates = 5

x_next_candidates = best_x + np.random.normal(0, sigma, size=(num_candidates, 4))
# Ensure all points are within [0,1]
x_next_candidates = np.clip(x_next_candidates, 0, 1)

print("Candidate next points to evaluate:")
print(x_next_candidates)

Candidate next points to evaluate:
[[0.22600824 0.814252   0.86968573 0.87510854]
 [0.2206631  0.84983133 0.83632751 0.92558874]
 [0.18323785 0.81609301 0.86437678 0.88179835]
 [0.25093763 0.86850565 0.89044218 0.86346306]
 [0.23829148 0.84381201 0.87201912 0.84790825]]


Week-03

In [26]:
X_init = data_in           # shape (n_samples, 4)
y_init = data_out          # shape (n_samples,)

# --- Add the two new data points ---
new_points = np.array([
    [0.232877,0.841416,0.883342,0.879464],
    [0.245154,0.843081,0.898729,0.883391]
])

new_outputs = np.array([
    1091.3271430129832,
    1195.7279770056589
])


# Combine all data
X_all = np.vstack([X_init, new_points])
y_all = np.concatenate([y_init, new_outputs])

# --- Fit updated Gaussian Process ---
kernel = Matern(nu=2.5)
gp = GaussianProcessRegressor(
    kernel=kernel,
    alpha=1e-8,                # smaller noise term for precision
    normalize_y=True,
    n_restarts_optimizer=5     # more robust kernel fitting
)
gp.fit(X_all, y_all)

# --- Define acquisition function (UCB variant) ---
def surrogate_neg_ucb(x, kappa=2.0):
    mean, std = gp.predict(x.reshape(1, -1), return_std=True)
    return -(mean + kappa * std)

# --- Search bounds for normalized inputs ---
bounds = [(0, 1), (0, 1), (0, 1), (0, 1)]

# --- Optimize acquisition function for next sampling point ---
best_x, best_val = None, float('inf')
for _ in range(10):
    x0 = np.random.rand(4)
    res = minimize(surrogate_neg_ucb, x0=x0, bounds=bounds, method='L-BFGS-B')
    if res.fun < best_val:
        best_val = res.fun
        best_x = res.x

x_next = best_x

print("Suggested next point to evaluate:", x_next)

# --- Optionally: generate a few local perturbations for fine exploration ---
sigma = 0.01
num_candidates = 5
x_next_candidates = x_next + np.random.normal(0, sigma, size=(num_candidates, 4))
x_next_candidates = np.clip(x_next_candidates, 0, 1)

print("\nLocal candidate points for fine-tuning:")
print(x_next_candidates)

res_formatted = [f"{r:.6f}" for r in x_next]
result = "-".join(res_formatted)
print(result)

x_next_6dp = np.round(x_next, 6)
x_next_6dp

Suggested next point to evaluate: [0.47157601 0.15269642 0.1269874  0.08391561]

Local candidate points for fine-tuning:
[[0.46826053 0.14755267 0.12489952 0.08184629]
 [0.47493817 0.15651773 0.12273761 0.07886842]
 [0.47397061 0.16243226 0.10157407 0.08313209]
 [0.48095705 0.15825232 0.125312   0.08975855]
 [0.45477953 0.14869936 0.12904835 0.08116749]]
0.471576-0.152696-0.126987-0.083916


array([0.471576, 0.152696, 0.126987, 0.083916])

week-04

In [27]:
X_new = np.array([
    [0.232877,0.841416,0.883342,0.879464],
    [0.245154,0.843081,0.898729,0.883391],
    [0.237286,0.897828,0.947445,0.897134]
])
y_new = np.array([ 
    1091.3271430129832,
    1195.7279770056589,
    1880.5766271350337
])

# Combine all data
X_all= np.vstack((data_in, X_new))
y_all = np.concatenate((data_out, y_new))


eps= 1e-20
signs = np.sign(y_all)
signs[signs == 0] = 1.0
y_trans = signs * np.log10(np.abs(y_all) + eps)


kernel = Matern(length_scale = 0.1, nu=2.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp.fit(X_all, y_trans)

def acquisition_ei(X, gp, y_best, xi=0.05):
    mu, sigma = gp.predict(X, return_std=True)
    sigma = sigma.reshape(-1, 1)
    mu = mu.reshape(-1, 1)
    imp = mu - y_best - xi
    Z = imp / (sigma + 1e-9)
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return ei.ravel()

grid_size = 100
margin = 0.03
n = 6

# Create 1D ranges
x1 = np.linspace(margin, 1-margin, n)
x2 = np.linspace(margin, 1-margin, n)
x3 = np.linspace(margin, 1-margin, n)
x4 = np.linspace(margin, 1-margin, n)

# Create 4D meshgrid
X1, X2, X3, X4 = np.meshgrid(
    x1, x2, x3, x4, 
    indexing='ij'
)

# Convert into candidate points
X_candidates = np.vstack([
    X1.ravel(),
    X2.ravel(),
    X3.ravel(),
    X4.ravel()
]).T

#bounds = [(X_all[:,i].min(), X_all[:,i].max()) for i in range(4)]
#num_candidates = 5000
#X_candidates = np.column_stack([
#    np.random.uniform(b[0], b[1], num_candidates) for b in bounds
#])

# --- 6. Compute EI across the grid ---
y_best = np.max(y_trans)
acq_values = acquisition_ei(X_candidates, gp, y_best, xi=1.0)

# --- 7. Select the next point ---
next = X_candidates[np.argmax(acq_values)]
best_ei = np.max(acq_values)

# --- 8. Display results with precision ---
print(f"[{next[0]:.6f}, {next[1]:.6f}, {next[2]:.6f}, {next[3]:.6f}]")

print(cf.format_inputdata(next))

[0.594000, 0.970000, 0.970000, 0.970000]
0.594000-0.970000-0.970000-0.970000


[0.406000, 0.970000, 0.970000, 0.970000] for 0.05 <br/>
[0.782000, 0.970000, 0.970000, 0.970000] for 1.0 <br/>
[0.782000, 0.970000, 0.970000, 0.970000] for 2.0 <br/>

Week-05

In [28]:
from sklearn.gaussian_process.kernels import ConstantKernel, WhiteKernel
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor

In [29]:
X_new = np.array([
    [0.232877,0.841416,0.883342,0.879464], #week-01 new input
    [0.245154,0.843081,0.898729,0.883391], #week-02 new input
    [0.237286,0.897828,0.947445,0.897134], #week-03 new input
    [0.594000, 0.970000, 0.970000, 0.970000] #week-04 new input
])
y_new = np.array([ 
    1091.3271430129832, #week-01 new output
    1195.7279770056589, #week-02 new output
    1880.5766271350337, #week-03 new output
    3735.477417437672 #week-04 new output
])

# Combine all data
X_init = np.vstack((data_in, X_new))
y_init = np.concatenate((data_out, y_new))


eps= 1e-20
signs = np.sign(y_init)
signs[signs == 0] = 1.0
y_trans = signs * np.log10(np.abs(y_init) + eps)

#kernel = Matern(length_scale = 0.1, nu=2.5)
kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=0.25, length_scale_bounds=(1e-2, 2.0), nu=2.5) \
         + WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-10, 1e-1))

#gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp = GaussianProcessRegressor(
    kernel=kernel, 
    alpha=1e-8, 
    normalize_y=True,
    n_restarts_optimizer=8,
    random_state=42
    )

gp.fit(X_init, y_trans)

rf = RandomForestRegressor(
    n_estimators=600, 
    random_state=42,
    bootstrap=True,
    max_features=1.0,
    min_samples_leaf=1,
    n_jobs=-1
    )

rf.fit(X_init, y_trans)

gbm_ens=[]
for k in range(12):
    gbm = GradientBoostingRegressor(
        n_estimators=350, 
        learning_rate=0.05, 
        max_depth=3, 
        random_state=100+k,
        subsample=0.8,
        min_samples_leaf=2
        )
    gbm.fit(X_init, y_trans)
    gbm_ens.append(gbm)



def acquisition_ei(X, gp, y_best, xi=0.01):
    mu, sigma = gp.predict(X, return_std=True)
    sigma = sigma.reshape(-1, 1)
    mu = mu.reshape(-1, 1)
    imp = mu - y_best - xi
    Z = imp / (sigma + 1e-9)
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return ei.ravel()

grid_size = 100
margin = 0.02
n=6
x1 = np.linspace(margin, 1 - margin, n)
x2 = np.linspace(margin, 1 - margin, n)
x3 = np.linspace(margin, 1-margin, n)
x4 = np.linspace(margin, 1-margin, n)

X1, X2, X3, X4 = np.meshgrid(
    x1, x2, x3, x4,
    indexing='ij'
)

# Convert into candidate points
X_candidates = np.vstack([
    X1.ravel(),
    X2.ravel(),
    X3.ravel(),
    X4.ravel()
]).T

# --- 6. Compute EI across the grid ---
y_best = np.max(y_trans)
acq_values = acquisition_ei(X_candidates, gp, y_best, xi=0.5)

# --- 7. Select the next point ---
next_point = X_candidates[np.argmax(acq_values)]
best_ei = np.max(acq_values)

# --- 8. Display results with precision ---
print(f"Next point using ei: [{next_point[0]:.6f}, {next_point[1]:.6f}, {next_point[2]:.6f}, {next_point[3]:.6f}]")

#res_formatted = [f"{r:.6f}" for r in next_point]
#result = "-".join(res_formatted)
print(cf.format_inputdata(next_point))


Next point using ei: [0.020000, 0.020000, 0.020000, 0.980000]
0.020000-0.020000-0.020000-0.980000


Week-06

In [30]:
X_new = np.array([
    [0.232877,0.841416,0.883342,0.879464], #week-01 new input
    [0.245154,0.843081,0.898729,0.883391], #week-02 new input
    [0.237286,0.897828,0.947445,0.897134], #week-03 new input
    [0.594000, 0.970000, 0.970000, 0.970000], #week-04 new input
    [0.020000, 0.020000, 0.020000, 0.980000] #week-05 new input
])
y_new = np.array([ 
    1091.3271430129832, #week-01 new output
    1195.7279770056589, #week-02 new output
    1880.5766271350337, #week-03 new output
    3735.477417437672, #week-04 new output
    139.06436844336912 #week-05 new output
])

# Combine all data
X_init = np.vstack((data_in, X_new))
y_init = np.concatenate((data_out, y_new))


eps= 1e-20
signs = np.sign(y_init)
signs[signs == 0] = 1.0
y_trans = signs * np.log10(np.abs(y_init) + eps)

#kernel = Matern(length_scale = 0.1, nu=2.5)
kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=0.25, length_scale_bounds=(1e-2, 2.0), nu=2.5) \
         + WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-10, 1e-1))

#gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, normalize_y=True)
gp = GaussianProcessRegressor(
    kernel=kernel, 
    alpha=1e-8, 
    normalize_y=True,
    n_restarts_optimizer=8,
    random_state=42
    )

gp.fit(X_init, y_trans)

rf = RandomForestRegressor(
    n_estimators=600, 
    random_state=42,
    bootstrap=True,
    max_features=1.0,
    min_samples_leaf=1,
    n_jobs=-1
    )

rf.fit(X_init, y_trans)

gbm_ens=[]
for k in range(12):
    gbm = GradientBoostingRegressor(
        n_estimators=350, 
        learning_rate=0.05, 
        max_depth=3, 
        random_state=100+k,
        subsample=0.8,
        min_samples_leaf=2
        )
    gbm.fit(X_init, y_trans)
    gbm_ens.append(gbm)



def acquisition_ei(X, gp, y_best, xi=0.01):
    mu, sigma = gp.predict(X, return_std=True)
    sigma = sigma.reshape(-1, 1)
    mu = mu.reshape(-1, 1)
    imp = mu - y_best - xi
    Z = imp / (sigma + 1e-9)
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return ei.ravel()

grid_size = 100
margin = 0.02
n=6
x1 = np.linspace(margin, 1 - margin, n)
x2 = np.linspace(margin, 1 - margin, n)
x3 = np.linspace(margin, 1-margin, n)
x4 = np.linspace(margin, 1-margin, n)

X1, X2, X3, X4 = np.meshgrid(
    x1, x2, x3, x4,
    indexing='ij'
)

# Convert into candidate points
X_candidates = np.vstack([
    X1.ravel(),
    X2.ravel(),
    X3.ravel(),
    X4.ravel()
]).T

# --- 6. Compute EI across the grid ---
y_best = np.max(y_trans)
acq_values = acquisition_ei(X_candidates, gp, y_best, xi=0.5)

# --- 7. Select the next point ---
next_point = X_candidates[np.argmax(acq_values)]
best_ei = np.max(acq_values)

# --- 8. Display results with precision ---
print(f"Next point using ei: [{next_point[0]:.6f}, {next_point[1]:.6f}, {next_point[2]:.6f}, {next_point[3]:.6f}]")

#res_formatted = [f"{r:.6f}" for r in next_point]
#result = "-".join(res_formatted)
print(cf.format_inputdata(next_point))

Next point using ei: [0.980000, 0.788000, 0.980000, 0.980000]
0.980000-0.788000-0.980000-0.980000
